# Tutorial 10 — Extracellular-Matrix Turnover

This notebook explores survival, cohorts, maturation, inflammation, cross-links, and multicomponent mechanics.

In [ ]:
LANGUAGE = "en"
from pathlib import Path
import sys


def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Repository root not found. Open the notebook inside the repository.")


REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import biomechanics_tutorials
print("Repository:", REPOSITORY_ROOT)
print("Package:", Path(biomechanics_tutorials.__file__).resolve())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from biomechanics_tutorials.ecm_turnover import (
    SurvivalParameters, survival_fraction,
    simulate_cohort_turnover, simulate_homogenized_turnover,
    simulate_collagen_maturation, matrix_nominal_stress,
    MatrixComponent, simulate_multicomponent_ecm,
)

In [ ]:
age = np.linspace(0, 50, 300)
for law in [SurvivalParameters("exponential", 15), SurvivalParameters("weibull", 15, 0.7), SurvivalParameters("weibull", 15, 2.0)]:
    plt.plot(age, survival_fraction(age, law), label=f"{law.model}, k={law.shape}")
plt.xlabel("Age")
plt.ylabel("Survival")
plt.legend();

In [ ]:
time = np.linspace(0, 50, 501)
production = np.log(2) / 10
cohort = simulate_cohort_turnover(time, production, SurvivalParameters(half_life=10))
homogenized = simulate_homogenized_turnover(time, production, 10)
plt.plot(time, cohort["mass"], label="cohorts")
plt.plot(time, homogenized["mass"], "--", label="homogenized")
plt.legend();

In [ ]:
time = np.linspace(0, 80, 801)
result = simulate_collagen_maturation(
    time,
    stress_protocol=lambda t: 1.0 if t < 10 else 1.3,
    inflammation_protocol=lambda t: 0.8 if 30 <= t <= 45 else 0.0,
)
plt.plot(time, result["mature"], label="mature collagen")
plt.plot(time, result["crosslinks"], label="cross-links")
plt.plot(time, result["effective_stiffness"], label="stiffness")
plt.legend();

In [ ]:
stress = matrix_nominal_stress(1.10, 1.0, result["mature"], result["crosslinks"], 1.05)
plt.plot(time, stress)
plt.xlabel("Time")
plt.ylabel("Nominal stress at fixed stretch");

In [ ]:
components = [
    MatrixComponent("Elastin", 0.8, 0.0003, 800, 0.6),
    MatrixComponent("Collagen I", 1.0, np.log(2)/25, 25, 2.4, 1.05, 0.7, 0.15),
    MatrixComponent("Collagen III", 0.45, np.log(2)*0.45/12, 12, 1.2, 1.02, 1.0, 0.3),
    MatrixComponent("Proteoglycan", 0.3, np.log(2)*0.3/6, 6, 0.25),
]
multi = simulate_multicomponent_ecm(time, components, stress_protocol=lambda t: 1.0 if t < 15 else 1.25)
for i, name in enumerate(multi["names"]):
    plt.plot(time, multi["mass"][:, i], label=name)
plt.legend();

In [ ]:
print("Final mature collagen:", result["mature"][-1])
print("Final cross-link fraction:", result["crosslinks"][-1])
print("Final effective stiffness:", result["effective_stiffness"][-1])